In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyproj import Geod

In [2]:
fraud_train= pd.read_csv('fraudTrain.csv')
fraud_test= pd.read_csv('fraudTest.csv')

fraud_train['source'] = 'train'
fraud_test['source'] = 'test'
fraud_train['orig_index'] = fraud_train.index
fraud_test['orig_index'] = fraud_test.index

fraud_train['trans_date_trans_time'] = pd.to_datetime(fraud_train['trans_date_trans_time'], errors='coerce')
fraud_test['trans_date_trans_time'] = pd.to_datetime(fraud_test['trans_date_trans_time'], errors='coerce')

test_start = fraud_test['trans_date_trans_time'].min()
test_end = test_start + pd.DateOffset(months=1)   # first calendar month

fraud_test_1m = fraud_test[
    (fraud_test['trans_date_trans_time'] >= test_start) &
    (fraud_test['trans_date_trans_time'] < test_end)
].copy()

print("fraudTrain rows:", len(fraud_train))
print("fraudTest total rows:", len(fraud_test))
print("fraudTest first-month rows:", len(fraud_test_1m))
print("Test window:", test_start, "to", test_end)


fraudTrain rows: 1296675
fraudTest total rows: 555719
fraudTest first-month rows: 87645
Test window: 2020-06-21 12:14:25 to 2020-07-21 12:14:25


In [3]:
fraud_all = pd.concat([fraud_train, fraud_test_1m], axis=0)
fraud_all = fraud_all.reset_index(drop=True)
fraud_all['row_id_combined'] = fraud_all.index

In [4]:
fraud_all.columns = ['Unnamed', 'Transaction Date', 'Credit Card Number', 'Merchant', 'Category', 'Amount', 'First Name', 'Last Name', 'Gender', 'Street', 'City', 'State', 'Zip', 'Latitude', 'Longitude', 'City Population', 'Job', 'DOB', 'trans_num', 'Unix Time', 'Merchant Latitude', 'Merchant Longitude', 'Is Fraud', 'Source', 'Original Index', 'Row ID Combined']

In [5]:
locational_df = fraud_all[['Credit Card Number', 'State', 'Latitude', 'Longitude', 'trans_num','Merchant Latitude', 'Merchant Longitude', 'Is Fraud', 'Source', 'Original Index']].copy()
locational_fraud = locational_df.loc[locational_df['Is Fraud'] == 1].reset_index().drop(columns = 'index')
locational_not_fraud = locational_df.loc[locational_df['Is Fraud'] == 0].reset_index().drop(columns = 'index')

# list of all 50 states + DC
unique_states_df = locational_fraud['State'].unique()

# all fraudulent transactions grouped by state
state_fraud = {state: df.reset_index(drop=True) 
             for state, df in locational_fraud.groupby('State')}


## **Fraud Rate by State**

In [6]:
# fraudulent transactions in given state out of all transactions in that state
fraud_rate = {state: len(state_fraud[state]) / len(fraud_all.loc[fraud_all['State'] == state]) for state in state_fraud}

fraud_rate_df = pd.DataFrame(list(fraud_rate.items()), columns=['State', 'Fraud Rate'])
fraud_rate_df.sort_values(by='Fraud Rate', ascending=False, inplace=True)
fraud_rate_df.reset_index(drop=True, inplace=True)

In [7]:
# DE 100% fraud rate, so to not skew the graph, remove from the rest of the data
no_DE_df = fraud_rate_df[fraud_rate_df['State'] != 'DE'].reset_index(drop=True)

In [8]:
# plt.figure(figsize=(15,6))
# plt.bar(x=no_DE_df['State'], height=no_DE_df['Fraud Rate'], width=0.6)  
# plt.margins(x=0.007)
# plt.ylabel('Rate of Fraudulent Transactions')
# plt.xlabel('State')
# plt.title('Rate of Fraudulent Transactions by State')
# plt.show()

In [9]:
# import plotly.express as px

# fig = px.choropleth(
#     no_DE_df,
#     locations="State",
#     locationmode="USA-states",
#     color="Fraud Rate",
#     scope="usa",
#     color_continuous_scale="Reds"
# )

# fig.update_layout(title="Fraud Rate by State")
# fig.show()

In [11]:
fraud_rate_df.describe()
no_DE_df.describe()

top_states = fraud_rate_df.head(10)
bottom_states = fraud_rate_df.tail(10)
# print(top_states, '\n\n', bottom_states)

state_counts = locational_df.groupby('State')['Is Fraud'].count()
state_counts 

locational_df["State Fraud Rate"] = locational_df["State"].map(fraud_rate)


## **Distance from transaction to merchant**

In [48]:
# avg_dist_df = pd.DataFrame(locational_df.groupby('State')['Distance (m)'].mean(), columns = ['Average Distance (m)'])
# avg_dist_df 

In [12]:
import plotly.express as px


geod = Geod(ellps="WGS84")

_, _, distances = geod.inv(
    locational_df['Longitude'].values,
    locational_df['Latitude'].values,
    locational_df['Merchant Longitude'].values,
    locational_df['Merchant Latitude'].values
)

locational_df['Distance (km)'] = distances/1000
# locational_df.head()


In [13]:
geod = Geod(ellps="WGS84")

_, _, distances = geod.inv(
    locational_fraud['Longitude'].values,
    locational_fraud['Latitude'].values,
    locational_fraud['Merchant Longitude'].values,
    locational_fraud['Merchant Latitude'].values
)

locational_fraud['Distance (km)'] = distances/1000
# locational_fraud.head()



In [14]:
# # map of fraud transactions by location, colored by distance

# df_sample = locational_fraud.sample(1000)

# fig = px.scatter_mapbox(
#     # df_sample,
#     locational_fraud,
#     lat="Latitude",
#     lon="Longitude",
#     size="Distance (km)",                 
#     size_max=7,                         
#     color="Distance (km)",                
#     color_continuous_scale="Turbo",      
#     hover_data=["State", "Distance (km)"],
#     zoom=3,
#     height=600
# )

# fig.update_layout(mapbox_style="carto-positron")

# fig.show()

In [15]:
geod = Geod(ellps="WGS84")

_, _, distances = geod.inv(
    locational_not_fraud['Longitude'].values,
    locational_not_fraud['Latitude'].values,
    locational_not_fraud['Merchant Longitude'].values,
    locational_not_fraud['Merchant Latitude'].values
)

locational_not_fraud['Distance (km)'] = distances/1000
# locational_not_fraud.head()



In [16]:
# # map of non-fraud transactions by location, colored by distance
# # only sample because total data too large 


# df_sample = locational_not_fraud.sample(1000)

# fig = px.scatter_mapbox(
#     df_sample,
#     lat="Latitude",
#     lon="Longitude",
#     size="Distance (km)",                 
#     size_max=7,                         
#     color="Distance (km)",                
#     color_continuous_scale="Turbo",      
#     hover_data=["State", "Distance (km)"],
#     zoom=3,
#     height=600
# )

# fig.update_layout(mapbox_style="carto-positron")

# fig.show()

In [17]:
# # distance distributions
# plt.figure(figsize=(10,6))

# sns.kdeplot(
#     data=locational_df,
#     x="Distance (km)",
#     hue="Is Fraud",
#     log_scale=True
# )

# plt.title("Distance distribution: fraud vs non-fraud")
# plt.show()

In [18]:
# summary statistics

# locational_df.groupby("Is Fraud")["Distance (km)"].describe()


In [19]:
# visualize fraud prob. vs. distance

# locational_df["Distance Bin"] = pd.qcut(locational_df["Distance (km)"], 20)
# distance_fraud = locational_df.groupby("Distance Bin")["Is Fraud"].mean()
# locational_df['Distance Bin'][0:21]


In [20]:
# distance_fraud.plot(xlabel="Distance (km)", ylabel="Fraud Probability", title="Fraud Probability vs. Distance (km)", fontsize=6)

In [21]:
# classify thresholds for transactions being local, regional, long-distance, international
# compare rates of fraud across these categories

# locational_df["Region"] = pd.cut(locational_df["Distance (km)"], bins=[0, 50, 500, 5000, float("inf")], labels=["Local", "Regional", "Long-Distance", "International"])  
# locational_df.head()

In [22]:
'''
Distance groups:

(0,50] local
(50,500] regional
(500, 5000] long-distance
(5000, inf) international

'''

locational_df["Distance Bin"] = pd.cut(
    locational_df["Distance (km)"],
    bins=[0, 50, 500, 5000, float("inf")],

)

# fraud rate for each region
distance_fraud = locational_df.groupby("Distance Bin")["Is Fraud"].mean()
locational_df['Bin Fraud Rate'] = locational_df['Distance Bin'].map(distance_fraud)
locational_df


C:\Users\reina\AppData\Local\Temp\ipykernel_49944\1428985699.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  distance_fraud = locational_df.groupby("Distance Bin")["Is Fraud"].mean()


,Credit Card Number,State,Latitude,Longitude,trans_num,Merchant Latitude,Merchant Longitude,Is Fraud,Source,Original Index,State Fraud Rate,Distance (km),Distance Bin,Bin Fraud Rate
0,2703186189652095,NC,36.0788,-81.1781,0b242abb623afc578575680df30655b9,36.011293,-82.048315,0,train,0,0.005826,78.773821,"(50.0, 500.0]",0.00572
1,630423337322,WA,48.8878,-118.2105,1f76529f8574734946361c461b024d99,49.159047,-118.186462,0,train,1,0.005449,30.216618,"(0.0, 50.0]",0.00553
2,38859492057661,ID,42.1808,-112.2620,a1a22d70485983eac12b5b88dad1cf95,43.150704,-112.154481,0,train,2,0.001843,108.102912,"(50.0, 500.0]",0.00572
3,3534093764340240,MT,46.2306,-112.1138,6b849c168bdad6f867558c3793159a81,47.034331,-112.561071,0,train,3,0.003515,95.685115,"(50.0, 500.0]",0.00572
4,375534208663984,VA,38.4207,-79.4629,a41d7549acf90789359a9aa5346dcb46,38.674999,-78.632459,0,train,4,0.006760,77.702395,"(50.0, 500.0]",0.00572
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1384315,6011366578560244,PA,40.5046,-77.7186,28dbfcfd6288b406e125154d26a8048b,41.103110,-77.601875,0,test,87640,0.005370,67.190464,"(50.0, 500.0]",0.00572
1384316,2264937662466770,TX,29.3641,-98.4924,a057b7eecf1683d1109afd59d6478d6a,29.993934,-98.163099,0,test,87641,0.004851,76.747307,"(50.0, 500.0]",0.00572
1384317,30082025922891,FL,26.1184,-81.7361,ba8f31627906ee914d3150b3557ebe03,25.973459,-81.848113,0,test,87642,0.006431,19.583677,"(0.0, 50.0]",0.00553
1384318,30235438713303,WV,38.5072,-81.8900,014019287afe680e698414b3ca38dc09,37.583384,-82.006009,0,test,87643,0.005330,103.045475,"(50.0, 500.0]",0.00572


In [23]:
# locational_fraud['Category'] = pd.cut(locational_fraud["Distance (km)"], bins=[0, 50, 500, 5000, float("inf")], labels=["Local", "Regional", "Long-Distance", "International"])
# locational_not_fraud['Category'] = pd.cut(locational_not_fraud["Distance (km)"], bins=[0, 50, 500, 5000, float("inf")], labels=["Local", "Regional", "Long-Distance", "International"])    
# locational_fraud.drop(columns=['Zip', 'Latitude', 'Longitude', 'Merchant Latitude', 'Merchant Longitude'], inplace = True)
# locational_not_fraud.drop(columns=['Zip', 'Latitude', 'Longitude', 'Merchant Latitude', 'Merchant Longitude'], inplace = True)
# locational_fraud.head()

In [24]:
# transform to log scale

import math
# log_distance = log(distance + 1)
locational_df["Log Distance (km)"] = np.log1p(locational_df["Distance (km)"])
locational_df.head()

,Credit Card Number,State,Latitude,Longitude,trans_num,Merchant Latitude,Merchant Longitude,Is Fraud,Source,Original Index,State Fraud Rate,Distance (km),Distance Bin,Bin Fraud Rate,Log Distance (km)
0,2703186189652095,NC,36.0788,-81.1781,0b242abb623afc578575680df30655b9,36.011293,-82.048315,0,train,0,0.005826,78.773821,"(50.0, 500.0]",0.00572,4.379195
1,630423337322,WA,48.8878,-118.2105,1f76529f8574734946361c461b024d99,49.159047,-118.186462,0,train,1,0.005449,30.216618,"(0.0, 50.0]",0.00553,3.440951
2,38859492057661,ID,42.1808,-112.2620,a1a22d70485983eac12b5b88dad1cf95,43.150704,-112.154481,0,train,2,0.001843,108.102912,"(50.0, 500.0]",0.00572,4.692292
3,3534093764340240,MT,46.2306,-112.1138,6b849c168bdad6f867558c3793159a81,47.034331,-112.561071,0,train,3,0.003515,95.685115,"(50.0, 500.0]",0.00572,4.571459
4,375534208663984,VA,38.4207,-79.4629,a41d7549acf90789359a9aa5346dcb46,38.674999,-78.632459,0,train,4,0.006760,77.702395,"(50.0, 500.0]",0.00572,4.365674


In [25]:
# compare transaction distance to typical (average) distance for that card
# first calculate average distance for cardholder

locational_df['Avg Card Dist (km)'] = locational_df.groupby('Credit Card Number')['Distance (km)'].transform('mean')
locational_df['Dist Ratio'] = locational_df['Distance (km)']/locational_df['Avg Card Dist (km)']
# locational_df.head()


In [26]:
final_df = locational_df[['trans_num', 'State Fraud Rate', 'Distance (km)', 'Avg Card Dist (km)', 'Dist Ratio', 'Log Distance (km)', 'Distance Bin',  'Bin Fraud Rate', 'Source', 'Original Index']]
final_df.head()

,trans_num,State Fraud Rate,Distance (km),Avg Card Dist (km),Dist Ratio,Log Distance (km),Distance Bin,Bin Fraud Rate,Source,Original Index
0,0b242abb623afc578575680df30655b9,0.005826,78.773821,77.786312,1.012695,4.379195,"(50.0, 500.0]",0.00572,train,0
1,1f76529f8574734946361c461b024d99,0.005449,30.216618,71.848365,0.420561,3.440951,"(0.0, 50.0]",0.00553,train,1
2,a1a22d70485983eac12b5b88dad1cf95,0.001843,108.102912,74.164986,1.457600,4.692292,"(50.0, 500.0]",0.00572,train,2
3,6b849c168bdad6f867558c3793159a81,0.003515,95.685115,71.275187,1.342474,4.571459,"(50.0, 500.0]",0.00572,train,3
4,a41d7549acf90789359a9aa5346dcb46,0.006760,77.702395,75.279400,1.032187,4.365674,"(50.0, 500.0]",0.00572,train,4


In [27]:
feat_train = final_df[final_df['Source'] == 'train'].copy()
feat_test_1m = final_df[final_df['Source'] == 'test'].copy()

# Optional: restore original file order
feat_train = feat_train.sort_values('Original Index')
feat_test_1m = feat_test_1m.sort_values('Original Index')

feat_train = feat_train.drop(columns=['Source', 'Original Index'])
feat_test_1m = feat_test_1m.drop(columns=['Source', 'Original Index'])

print("feat_train shape:", feat_train.shape)
print("feat_test_1m shape:", feat_test_1m.shape)

feat_train shape: (1296675, 8)
feat_test_1m shape: (87645, 8)


In [28]:
order_train = pd.read_csv('C:\\Users\\reina\\Downloads\\Math 168\\feat_train.csv')

In [29]:
order_test = pd.read_csv('C:\\Users\\reina\\Downloads\\Math 168\\feat_test1m.csv')

In [30]:
print(' Length of features train: ', len(feat_train), '\n', 
      "Length of Jessica's train: ", len(order_train), '\n',
      'Length of features test_1m: ', len(feat_test_1m), '\n', 
      "Length of Jessica's test_1m: ", len(order_test) )

 Length of features train:  1296675 
 Length of Jessica's train:  1296675 
 Length of features test_1m:  87645 
 Length of Jessica's test_1m:  87645


In [ ]:
# feat_train.to_csv('distance_train.csv')
# feat_test_1m.to_csv('distance_test_1m.csv')

## filler dataset

In [31]:
filler_df = locational_df[['trans_num', 'State Fraud Rate', 'Distance (km)', 'Avg Card Dist (km)', 'Dist Ratio', 'Log Distance (km)', 'Distance Bin',  'Bin Fraud Rate', 'Is Fraud', 'Source', 'Original Index']]
filler_df.head()

,trans_num,State Fraud Rate,Distance (km),Avg Card Dist (km),Dist Ratio,Log Distance (km),Distance Bin,Bin Fraud Rate,Is Fraud,Source,Original Index
0,0b242abb623afc578575680df30655b9,0.005826,78.773821,77.786312,1.012695,4.379195,"(50.0, 500.0]",0.00572,0,train,0
1,1f76529f8574734946361c461b024d99,0.005449,30.216618,71.848365,0.420561,3.440951,"(0.0, 50.0]",0.00553,0,train,1
2,a1a22d70485983eac12b5b88dad1cf95,0.001843,108.102912,74.164986,1.457600,4.692292,"(50.0, 500.0]",0.00572,0,train,2
3,6b849c168bdad6f867558c3793159a81,0.003515,95.685115,71.275187,1.342474,4.571459,"(50.0, 500.0]",0.00572,0,train,3
4,a41d7549acf90789359a9aa5346dcb46,0.006760,77.702395,75.279400,1.032187,4.365674,"(50.0, 500.0]",0.00572,0,train,4


In [32]:
filler_train = filler_df[filler_df['Source'] == 'train'].copy()
filler_test_1m = filler_df[filler_df['Source'] == 'test'].copy()

# Optional: restore original file order
filler_train = filler_train.sort_values('Original Index')
filler_test_1m = filler_test_1m.sort_values('Original Index')

filler_train = filler_train.drop(columns=['Source', 'Original Index'])
filler_test_1m = filler_test_1m.drop(columns=['Source', 'Original Index'])

In [38]:
# filler_train.to_csv('filler_train.csv')
# filler_test_1m.to_csv('filler_test_1m.csv')

In [36]:
features_test = pd.read_csv('features_test.csv')
features_test = features_test.drop(columns=['Unnamed: 0'])
features_test['is_fraud'] = filler_test_1m['Is Fraud'].values
features_test

,trans_num,amt,lat,long,city_pop,unix_time,merch_lat,merch_long,log_amt,merchant_txn_count_past,...,TXScore_ST_birank,CCHScore_MT_birank,MCScore_MT_birank,TXScore_MT_birank,CCHScore_LT_birank,MCScore_LT_birank,TXScore_LT_birank,card_birank,merchant_birank,is_fraud
0,2da90c7d74bd46a0caf3777415b3ebd3,2.86,33.9659,-80.9355,333497,1371816865,33.986391,-81.200714,1.350667,1816.0,...,0.000512,0.000166,0.000096,0.000014,0.000140,0.000117,0.000002,0.001176,0.001392,0
1,324cc204407e99f51b0d6ca0055005e7,29.84,40.3207,-110.4360,302,1371816873,39.450498,-109.960431,3.428813,1825.0,...,0.001081,0.000239,0.000139,0.000013,0.000246,0.000203,0.000003,0.001285,0.001455,0
2,c81755dbbbea9d5c77f094348a7579be,41.28,40.6729,-73.5365,34496,1371816893,40.495810,-74.196111,3.744314,1716.0,...,0.000394,0.000182,0.000070,0.000011,0.000326,0.000175,0.000003,0.001588,0.001447,0
3,2159175b9efe66dc301f149d3d5abf8c,60.05,28.5697,-80.8191,54767,1371816915,28.812398,-80.883061,4.111693,1629.0,...,0.000653,0.000148,0.001310,0.000066,0.000148,0.000186,0.000003,0.001142,0.001340,0
4,57ff021bd3f328f8738bb535c302a31b,3.19,44.2529,-85.0170,1126,1371816917,44.959148,-85.884734,1.432701,831.0,...,0.000466,0.000100,0.000054,0.000011,0.000220,0.000130,0.000004,0.001308,0.000916,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87640,28dbfcfd6288b406e125154d26a8048b,102.10,40.5046,-77.7186,4653,1374408807,41.103110,-77.601875,4.635699,2611.0,...,0.000880,0.000201,0.000252,0.000015,0.000246,0.001569,0.000011,0.001269,0.001873,0
87641,a057b7eecf1683d1109afd59d6478d6a,7.95,29.3641,-98.4924,1595797,1374408810,29.993934,-98.163099,2.191654,1917.0,...,0.001105,0.000180,0.000183,0.000014,0.000294,0.000217,0.000004,0.001400,0.001383,0
87642,ba8f31627906ee914d3150b3557ebe03,123.11,26.1184,-81.7361,276002,1374408812,25.973459,-81.848113,4.821168,2615.0,...,0.000809,0.000108,0.000238,0.000015,0.000170,0.000298,0.000003,0.001134,0.001659,0
87643,014019287afe680e698414b3ca38dc09,811.45,38.5072,-81.8900,5512,1374408816,37.583384,-82.006009,6.700054,1282.0,...,0.000928,0.000403,0.000156,0.000020,0.000281,0.000499,0.000007,0.001433,0.001146,0


In [37]:
features_test.to_csv('Merged_Test_is_fraud.csv')